# Load MMLU AUX TRAIN autolabelled (so we can differentiate subjects)

This notebook processes the MMLU AUX dataset, focusing on selected subjects. It includes steps to load, analyze, and export data for further use.

In [1]:
from datasets import load_dataset
import pandas as pd
import os
import pickle
from math import ceil

# Define constants
DATASET_NAME = 'kz919/mmlu-auxiliary-train-auto-labelled'
SPLIT = 'train'
SUBJECTS = ['college_biology', 'management', 'college_chemistry', 'global_facts', 'medical_genetics']
VAL_FRAC = 0.1  # Fraction to reserve for validation
RANDOM_STATE = 42
SEPARATOR = "---"

c:\Users\bjorn\.conda\envs\mmlu1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load the dataset and convert to DataFrame
def load_and_prepare_dataset():
    print("Loading dataset...")
    ds = load_dataset(DATASET_NAME, split=SPLIT)
    print(f"Loaded dataset with {len(ds)} samples.")

    # Inspect the dataset schema
    print("Dataset schema:")
    print(ds.features)

    # Convert to DataFrame with error handling
    try:
        df = ds.to_pandas()
        print(f"Converted to DataFrame with shape {df.shape}.")
    except Exception as e:
        print("Error during conversion to DataFrame:", e)
        print("Attempting to load a subset of the dataset...")
        # Load a smaller subset for debugging
        df = ds.select(range(100)).to_pandas()
        print(f"Loaded subset with shape {df.shape}.")

    return df

def ensure_qa_column(df):
    if 'qa' not in df.columns:
        df['qa'] = df['question'] + "\n" + df['choices']
    return df

df = load_and_prepare_dataset()
df = ensure_qa_column(df)

Loading dataset...
Loaded dataset with 99842 samples.
Dataset schema:
{'question': Value(dtype='string', id=None), 'subject': Value(dtype='string', id=None), 'choices': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'answer': Value(dtype='int64', id=None), 'task': Value(dtype='string', id=None)}
Loaded dataset with 99842 samples.
Dataset schema:
{'question': Value(dtype='string', id=None), 'subject': Value(dtype='string', id=None), 'choices': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'answer': Value(dtype='int64', id=None), 'task': Value(dtype='string', id=None)}
Converted to DataFrame with shape (99842, 5).
Converted to DataFrame with shape (99842, 5).


In [4]:
# Quick counts for the selected subjects
def print_subject_counts(df, subjects):
    print("Subject counts:")
    for subject in subjects:
        count = df['task'].value_counts().get(subject, 0)
        print(f"{subject}: {count}")

print_subject_counts(df, SUBJECTS)

Subject counts:
college_biology: 4156
management: 2066
college_chemistry: 490
global_facts: 334
medical_genetics: 76


In [5]:
# Export subject-specific CSV files
def export_subject_csvs(df, subjects):
    for subject in subjects:
        sub_df = df[df['task'] == subject].copy()
        if sub_df.empty:
            print(f"No data for subject '{subject}', skipping.")
            continue
        sub_df[['qa']].to_csv(f"{subject}_qa.csv", index=False)
        print(f"Exported {len(sub_df)} rows for '{subject}' to {subject}_qa.csv.")

export_subject_csvs(df, SUBJECTS)

Exported 4156 rows for 'college_biology' to college_biology_qa.csv.
Exported 2066 rows for 'management' to management_qa.csv.
Exported 490 rows for 'college_chemistry' to college_chemistry_qa.csv.
Exported 2066 rows for 'management' to management_qa.csv.
Exported 490 rows for 'college_chemistry' to college_chemistry_qa.csv.
Exported 334 rows for 'global_facts' to global_facts_qa.csv.
Exported 76 rows for 'medical_genetics' to medical_genetics_qa.csv.
Exported 334 rows for 'global_facts' to global_facts_qa.csv.
Exported 76 rows for 'medical_genetics' to medical_genetics_qa.csv.


In [6]:
# Create train/validation splits and save to CSVs
def create_train_val_splits(df, subjects, val_frac, random_state):
    for subject in subjects:
        sub_df = df[df['task'] == subject].copy()
        if sub_df.empty:
            print(f"No data for subject '{subject}', skipping.")
            continue
        sub_df = sub_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
        val_n = max(1, int(len(sub_df) * val_frac)) if len(sub_df) > 1 else 0
        train_df = sub_df.iloc[:len(sub_df) - val_n]
        val_df = sub_df.iloc[len(sub_df) - val_n:]
        train_df[['qa']].to_csv(f"{subject}_train.csv", index=False)
        val_df[['qa']].to_csv(f"{subject}_val.csv", index=False)
        print(f"{subject}: train={len(train_df)}, val={len(val_df)} -> saved to CSVs.")

create_train_val_splits(df, SUBJECTS, VAL_FRAC, RANDOM_STATE)

college_biology: train=3741, val=415 -> saved to CSVs.
management: train=1860, val=206 -> saved to CSVs.
college_chemistry: train=441, val=49 -> saved to CSVs.
global_facts: train=301, val=33 -> saved to CSVs.
medical_genetics: train=69, val=7 -> saved to CSVs.
management: train=1860, val=206 -> saved to CSVs.
college_chemistry: train=441, val=49 -> saved to CSVs.
global_facts: train=301, val=33 -> saved to CSVs.
medical_genetics: train=69, val=7 -> saved to CSVs.


In [7]:
# Convert train/val CSV files into binary `.bin` files
def convert_csv_to_bin(subjects, separator):
    for subject in subjects:
        for split in ['train', 'val']:
            csv_file = f"{subject}_{split}.csv"
            bin_file = f"{subject}_{split}.bin"

            if not os.path.exists(csv_file):
                print(f"CSV file {csv_file} not found, skipping conversion.")
                continue

            df = pd.read_csv(csv_file)
            data_with_separators = separator.join(df['qa'].tolist())

            with open(bin_file, 'wb') as f:
                pickle.dump(data_with_separators, f)

            print(f"Converted {csv_file} to {bin_file}, size: {os.path.getsize(bin_file)} bytes.")

convert_csv_to_bin(SUBJECTS, SEPARATOR)

Converted college_biology_train.csv to college_biology_train.bin, size: 15373438 bytes.
Converted college_biology_val.csv to college_biology_val.bin, size: 1723949 bytes.
Converted management_train.csv to management_train.bin, size: 13182670 bytes.
Converted management_val.csv to management_val.bin, size: 1451478 bytes.
Converted management_train.csv to management_train.bin, size: 13182670 bytes.
Converted management_val.csv to management_val.bin, size: 1451478 bytes.
Converted college_chemistry_train.csv to college_chemistry_train.bin, size: 659577 bytes.
Converted college_chemistry_val.csv to college_chemistry_val.bin, size: 66256 bytes.
Converted global_facts_train.csv to global_facts_train.bin, size: 1809156 bytes.
Converted global_facts_val.csv to global_facts_val.bin, size: 214776 bytes.
Converted college_chemistry_train.csv to college_chemistry_train.bin, size: 659577 bytes.
Converted college_chemistry_val.csv to college_chemistry_val.bin, size: 66256 bytes.
Converted global_fac

In [8]:
# Inspect a binary file
def inspect_binary_file(binary_file, separator, num_samples=3):
    if not os.path.exists(binary_file):
        print(f"Binary file {binary_file} not found.")
        return

    with open(binary_file, 'rb') as f:
        data = pickle.load(f)

    samples = data.split(separator)
    print("Sample from the binary file:")
    for i, sample in enumerate(samples[:num_samples]):
        print(f"Sample {i + 1}:")
        print(sample.strip())
        print(separator)

inspect_binary_file("college_biology_train.bin", SEPARATOR)

Sample from the binary file:
Sample 1:
["Domestic   horses now pull ploughs, race in the Kentucky Derby, and carry police. But early horses weren't tame   enough to perform these kinds of tasks. Scientists think the first interactions humans had with horses were far different from those today. Thousands of years ago, people killed the wild horses that lived around them for food. Over time, people began to catch the animals and raise them. This was the first step in domestication. As people began to tame and ride horses, they chose to keep those animals that had more desirable characteristics. For example, people may have chosen to keep horses that had a gentle personality so they could be ridden more easily. People who used horses to pull heavy loads would have chosen to keep stronger animals. Characteristics like strength are partly controlled by the animals' genes. So as the domesticated horses reproduced, they passed the characteristics on to their young. Each new generation of hous